# 1) CHARGEMENT DES PACKAGES NÉCESSAIRES

In [1]:
import io
import re
import gc
import os
import math
import time
from io import BytesIO
from typing import Optional, Dict, List, Iterable

import numpy as np
import pandas as pd

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

# Limite l'utilisation CPU des libs natives -> plus stable en RAM
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

'1'

# 2) DÉFINITION DES FONCTIONS

## 2.1 Lecture S3 : parquet -> DataFrame

In [2]:
def read_parquet_from_s3(
    bucket: str,
    key: str,
    *,
    endpoint_url: str,
    aws_access_key_id: str,
    aws_secret_access_key: str,
    verify: bool = False,
    columns: List[str] | None = None,   # <- PROJECTION dès la lecture
    engine: str | None = "pyarrow",
    max_attempts: int = 6,
    base_sleep: float = 1.0,
    fallback_download: bool = True
) -> pd.DataFrame:
    """
    Lit un Parquet depuis S3/MinIO en gérant les ralentissements 'SlowDown'.
    - columns: projection de colonnes (réduit fortement la RAM)
    - engine: 'pyarrow' recommandé
    - fallback_download: bascule sur 'download then read' si throttled
    """
    s3_client = boto3.client(
        "s3",
        endpoint_url=endpoint_url,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        verify=verify,
        config=Config(
            retries={"max_attempts": max_attempts, "mode": "adaptive"},
            max_pool_connections=16,
            read_timeout=180,
            connect_timeout=45,
        ),
    )

    attempt = 0
    while True:
        try:
            obj = s3_client.get_object(Bucket=bucket, Key=key)
            return pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=columns, engine=engine)
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            retryable = code in {"SlowDown", "SlowDownRead", "RequestTimeout", "503", "503SlowDown"}
            attempt += 1
            if retryable and attempt < max_attempts:
                sleep_s = min(base_sleep * (2 ** (attempt - 1)), 30)
                print(f"⚠️ {code}: tentative {attempt}/{max_attempts} — nouvelle tentative dans ~{sleep_s:.1f}s…")
                time.sleep(sleep_s)
                continue
            if fallback_download:
                print(f"ℹ️ Passage en mode 'download then read' (raison: {code or 'échec lecture stream'})")
                obj = s3_client.get_object(Bucket=bucket, Key=key)
                return pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=columns, engine=engine)
            raise


## 2.2 Détection de bases "répétées" (colonnes doublons logiques)

In [3]:
def _base_name(col: str) -> str:
    s = str(col).strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)      # retire '(...)' en fin
    s = re.sub(r"[_\-|./]+", " ", s)            # uniformise séparateurs
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

def find_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> Dict[str, List[str]]:
    groups: Dict[str, List[str]] = {}
    for c in df.columns:
        if c in exclude:
            continue
        base = _base_name(c)
        groups.setdefault(base, []).append(c)
    return {base: cols for base, cols in groups.items() if len(cols) > 1}

def report_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> pd.DataFrame:
    rep = find_repeated_columns(df, exclude=exclude)
    if not rep:
        print("✅ Aucune colonne répétée détectée.")
        return pd.DataFrame(columns=["base", "nb_colonnes", "colonnes"])
    print(f"🔎 Bases répétées détectées: {len(rep)}")
    rows = []
    for base, cols in sorted(rep.items()):
        print(f"  • {base} -> {cols}")
        rows.append({"base": base, "nb_colonnes": len(cols), "colonnes": cols})
    return (pd.DataFrame(rows)
              .sort_values(["nb_colonnes", "base"], ascending=[False, True])
              .reset_index(drop=True))


## 2.3 Extraction YEAR/MONTH depuis les colonnes date

In [4]:
def year_from_source_file(text: str) -> Optional[int]:
    s = str(text)
    m = re.search(r'(?<!\d)(0[1-9]|1[0-2])((?:19|20)\d{2})(?!\d)', s)
    if m:
        return int(m.group(2))
    m = re.search(r'(?:19|20)\d{2}', s)
    return int(m.group(0)) if m else None

def extract_years_series(src: pd.Series) -> pd.Series:
    return src.map(year_from_source_file)

def extract_years_unique(src: pd.Series) -> List[int]:
    years = extract_years_series(src).dropna().astype(int).unique().tolist()
    years.sort()
    return years

def year_month_from_source_file(s: pd.Series) -> pd.DataFrame:
    ss = s.astype("string").str.strip()
    m = ss.str.extract(r'^\s*(0[1-9]|1[0-2])\s*((?:19|20)\d{2})', expand=True)
    month = pd.to_numeric(m[0], errors="coerce")
    year  = pd.to_numeric(m[1], errors="coerce")
    return pd.DataFrame({"YEAR": year.astype("Int64"), "MONTH": month.astype("Int64")})

def year_month_from_date_collecte(s: pd.Series) -> pd.DataFrame:
    dt = pd.to_datetime(s, errors="coerce", utc=False)
    return pd.DataFrame({"YEAR": dt.dt.year.astype("Int64"), "MONTH": dt.dt.month.astype("Int64")})


## 2.4 Consolidation haute performance des colonnes répétées

In [5]:
def _filled_mask(block: pd.DataFrame, treat_zero_as_filled: bool) -> np.ndarray:
    n, m = block.shape
    filled = np.zeros((n, m), dtype=bool)
    for j, col in enumerate(block.columns):
        s = block[col]
        if pd.api.types.is_numeric_dtype(s):
            ok = s.notna().to_numpy()
            if not treat_zero_as_filled:
                ok &= (s.to_numpy() != 0)
        else:
            v = s.astype("object").to_numpy()
            ok = pd.notna(v)
            tmp = np.empty_like(v, dtype=object)
            if ok.any():
                tmp[ok] = np.char.lower(np.char.strip(v[ok].astype(str)))
                bad_ok = (tmp[ok] == "") | (tmp[ok] == "na") | (tmp[ok] == "nan") | (tmp[ok] == "none")
                bad = np.zeros(ok.shape, dtype=bool); bad[ok] = bad_ok
                ok &= ~bad
        filled[:, j] = ok
    return filled

def _coalesce_first_non_null(block: pd.DataFrame, treat_zero_as_filled: bool) -> pd.Series:
    if block.shape[1] == 1:
        s = block.iloc[:, 0]
        if pd.api.types.is_numeric_dtype(s) and not treat_zero_as_filled:
            return s.where(s.notna() & (s != 0))
        return s
    filled = _filled_mask(block, treat_zero_as_filled)
    any_filled = filled.any(axis=1)
    first_idx = filled.argmax(axis=1)
    arr = block.to_numpy(copy=False)
    rows = np.arange(arr.shape[0])
    out_vals = np.empty(arr.shape[0], dtype=object)
    out_vals[:] = pd.NA
    out_vals[any_filled] = arr[rows[any_filled], first_idx[any_filled]]
    return pd.Series(out_vals, index=block.index, dtype="object")

def _sum_numeric_block(block: pd.DataFrame, chunk: int = 50) -> pd.Series:
    """
    Somme ligne par ligne (axis=1) en traitant par paquets de colonnes (chunk)
    pour limiter les pics RAM.
    """
    idx = block.index
    out = pd.Series(0.0, index=idx, dtype="float64")
    ncols = block.shape[1]
    start = 0
    while start < ncols:
        subcols = block.columns[start:start+chunk]
        to_sum = {c: pd.to_numeric(block[c], errors="coerce") for c in subcols}
        part = pd.DataFrame(to_sum, index=idx).sum(axis=1, min_count=1).fillna(0.0)
        out = out.add(part, fill_value=0.0)
        start += chunk
    out = out.mask(out == 0.0, other=np.nan)  # si rien n'était présent
    return out

def _concat_block(block: pd.DataFrame, sep: str = " | ") -> pd.Series:
    # concat des valeurs renseignées seulement
    def _row_concat(row):
        vals = [str(v) for v in row if pd.notna(v) and str(v).strip() not in ("", "na", "nan", "none")]
        return sep.join(vals) if vals else pd.NA
    return block.apply(_row_concat, axis=1)

def consolidate_all_years_and_drop_sources(
    df: pd.DataFrame,
    source_col: str = "SOURCE_FICHIER",
    *,
    mode: str = "first_non_null",  # OPTI par défaut
    sep: str = " | ",
    treat_zero_as_filled: bool = True,
    new_name_fmt: str = "{base}",
    sum_chunk: int = 50
) -> pd.DataFrame:
    """
    Consolide les colonnes répétées par 'base' en UNE colonne finale par base,
    en travaillant **année par année** détectée dans `source_col`.
    Puis SUPPRIME les colonnes sources dupliquées.
    """
    if source_col not in df.columns:
        raise ValueError(f"Colonne {source_col!r} absente.")
    out = df.copy()

    repeated = find_repeated_columns(out, exclude=(source_col,))
    if not repeated:
        print("ℹ️ Aucune base répétée. Rien à consolider.")
        return out

    years = extract_years_unique(out[source_col])

    def _apply_mode(block: pd.DataFrame) -> pd.Series:
        if mode == "sum":
            return _sum_numeric_block(block, chunk=sum_chunk)
        elif mode == "first_non_null":
            return _coalesce_first_non_null(block, treat_zero_as_filled)
        elif mode == "concat":
            return _concat_block(block, sep=sep)
        else:
            raise ValueError(f"mode inconnu: {mode!r}")

    if years:
        out["_YEAR_"] = extract_years_series(out[source_col])
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            if new_col not in out.columns:
                out[new_col] = pd.NA
            for y in years:
                mask = out["_YEAR_"].eq(y)
                if not mask.any():
                    continue
                block = out.loc[mask, cols]
                out.loc[mask, new_col] = _apply_mode(block)
        out.drop(columns=["_YEAR_"], inplace=True, errors="ignore")
    else:
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            out[new_col] = _apply_mode(out[cols])

    cols_to_drop = sorted({c for cols in repeated.values() for c in cols})
    out.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    gc.collect()
    return out


## 2.5 Vérification d’exclusivité

In [6]:
def check_repeated_columns_exclusivity_by_year(
    df: pd.DataFrame,
    *,
    source_col: str = "SOURCE_FICHIER",
    treat_zero_as_filled: bool = True,
    show_examples: int = 10
) -> pd.DataFrame:
    if source_col not in df.columns:
        raise KeyError(f"Colonne {source_col!r} absente.")

    repeated = find_repeated_columns(df, exclude=(source_col,))
    if not repeated:
        print("✅ Aucune base répétée détectée — rien à vérifier.")
        return pd.DataFrame(columns=["base", "year", "violations", "rows_in_year", "columns"])

    years_series = extract_years_series(df[source_col])
    if years_series.isna().all():
        print("⚠️ Aucune année détectée — contrôle global.")
        years_series = pd.Series(["ALL"] * len(df), index=df.index)

    id_cols = [c for c in ["MATRICULE", "MATRICULE||CODE_ORGANISME", source_col] if c in df.columns] or [source_col]
    rows_summary = []
    any_violation = False

    for base, cols in repeated.items():
        block = df[cols]
        filled = _filled_mask(block, treat_zero_as_filled)
        nb_filled = filled.sum(axis=1)

        for year_val, inds in years_series.groupby(years_series).groups.items():
            mask_year = df.index.isin(inds)
            n_in_year = int(mask_year.sum())
            if n_in_year == 0:
                continue

            viol_mask = (nb_filled > 1) & mask_year
            n_viol = int(viol_mask.sum())

            rows_summary.append({
                "base": base,
                "year": year_val,
                "violations": n_viol,
                "rows_in_year": n_in_year,
                "columns": cols
            })

            if n_viol > 0:
                any_violation = True
                print(f"\n🚨 Exclusivité violée — base='{base}', année={year_val} : {n_viol} lignes en anomalie (sur {n_in_year}).")
                ex_cols = list(dict.fromkeys(id_cols + cols))
                print(df.loc[viol_mask, ex_cols].head(show_examples).to_string(index=False))

    summary = (pd.DataFrame(rows_summary)
                 .sort_values(["violations", "base", "year"], ascending=[False, True, True])
                 .reset_index(drop=True))

    if not any_violation:
        print("✅ Exclusivité respectée : au plus une colonne renseignée par base/année.")
    else:
        total_viol = int(summary["violations"].sum()) if not summary.empty else 0
        print(f"\nBilan: {total_viol} violations au total.")

    return summary


## 2.6 Jointure sur une année à gauche  : (YEAR, MONTH, MATRICULE)

In [7]:
def write_df_to_xlsx_chunked(
    df: pd.DataFrame,
    path: str,
    *,
    sheet_base: str = "data",
    chunk_rows: int = 1_000_000,
    engine: str = "xlsxwriter"
) -> None:
    n = len(df)
    if n == 0:
        with pd.ExcelWriter(path, engine=engine) as xw:
            df.to_excel(xw, sheet_name=sheet_base, index=False)
        return
    parts = math.ceil(n / chunk_rows)
    with pd.ExcelWriter(path, engine=engine) as xw:
        for i in range(parts):
            start = i * chunk_rows
            stop  = min((i + 1) * chunk_rows, n)
            sheet_name = sheet_base if i == 0 else f"{sheet_base}_part{i+1}"
            df.iloc[start:stop].to_excel(xw, sheet_name=sheet_name, index=False)

def _year_month_from_source_file(s: pd.Series) -> pd.DataFrame:
    ss = s.astype("string").str.strip()
    m = ss.str.extract(r'^\s*(0[1-9]|1[0-2])\s*((?:19|20)\d{2})', expand=True)
    return pd.DataFrame({
        "YEAR":  pd.to_numeric(m[1], errors="coerce").astype("Int64"),
        "MONTH": pd.to_numeric(m[0], errors="coerce").astype("Int64"),
    })

def _year_month_from_date_collecte(s: pd.Series) -> pd.DataFrame:
    dt = pd.to_datetime(s, errors="coerce", utc=False)
    return pd.DataFrame({
        "YEAR":  dt.dt.year.astype("Int64"),
        "MONTH": dt.dt.month.astype("Int64"),
    })

def _normalize_matricule_min(s: pd.Series) -> pd.Series:
    return (s.astype("string")
             .str.strip()
             .str.replace("\u00A0", "", regex=False))

def merge_union_for_year_with_suffix(
    df_left: pd.DataFrame,      # base 1 (gauche)
    df_right: pd.DataFrame,     # base 2 (droite)
    year: int,
    *,
    join_on_month: bool = True,
    suffix_left: str = "_b0",
    suffix_right: str = "_b1",
    dedup_left: bool = False,
    dedup_right: bool = True,
    export_unmatched_prefix: str | None = None,
    xlsx_chunk_rows: int = 1_000_000,
    xlsx_engine: str = "xlsxwriter",
    keep_left: List[str] | None = None,     # <- projection côté gauche
    keep_right: List[str] | None = None,    # <- projection côté droit
    max_xlsx_rows: int = 200_000            # <- plafond XLSX, sinon Parquet
) -> pd.DataFrame:

    # 1) Copies + normalisation matricule
    L = df_left.copy()
    R = df_right.copy()
    for d in (L, R):
        if "MATRICULE||CODE_ORGANISME" not in d.columns:
            raise KeyError("Il manque 'MATRICULE||CODE_ORGANISME' dans une des bases.")
        d["MATRICULE"] = _normalize_matricule_min(d["MATRICULE||CODE_ORGANISME"])

    # 2) YEAR/MONTH
    if "SOURCE_FICHIER" not in L.columns:
        raise KeyError("La base gauche doit contenir 'SOURCE_FICHIER'.")
    ymL = _year_month_from_source_file(L["SOURCE_FICHIER"])
    L["YEAR"], L["MONTH"] = ymL["YEAR"], ymL["MONTH"]

    if "DATE_COLLECTE" not in R.columns:
        raise KeyError("La base droite doit contenir 'DATE_COLLECTE'.")
    ymR = _year_month_from_date_collecte(R["DATE_COLLECTE"])
    R["YEAR"], R["MONTH"] = ymR["YEAR"], ymR["MONTH"]

    # 3) Filtre année
    L = L.loc[L["YEAR"].eq(year)].copy()
    R = R.loc[R["YEAR"].eq(year)].copy()

    # 4) Clés + projection (réduction RAM)
    keys = (["YEAR", "MONTH", "MATRICULE"] if join_on_month else ["YEAR", "MATRICULE"])

    if keep_left is not None:
        selL = list(set(keys + keep_left))
        L = L.loc[:, [c for c in selL if c in L.columns]]

    if keep_right is not None:
        selR = list(set(keys + keep_right))
        R = R.loc[:, [c for c in selR if c in R.columns]]

    # 5) Colonnes communes (hors clés) -> suffixes sélectifs
    overlap = sorted((set(L.columns) & set(R.columns)) - set(keys))
    if overlap:
        L.rename(columns={c: f"{c}{suffix_left}"  for c in overlap}, inplace=True)
        R.rename(columns={c: f"{c}{suffix_right}" for c in overlap}, inplace=True)

    # 6) Dédup éventuel
    if dedup_left:
        L = L.sort_values(keys).drop_duplicates(keys, keep="first")
    if dedup_right:
        R = R.sort_values(keys).drop_duplicates(keys, keep="first")

    # 7) Convertir texte non-clé en catégorie (fort gain RAM)
    for d in (L, R):
        for c in d.select_dtypes(include=["object", "string"]).columns:
            if c not in keys:
                try:
                    d[c] = d[c].astype("category")
                except Exception:
                    pass

    # 8) OUTER JOIN
    merged = L.merge(R, on=keys, how="outer", suffixes=("", ""), indicator=True)

    # 9) Indicateurs d’origine
    src_map = {"left_only": "base1", "right_only": "base2", "both": "both"}
    merged["IN_SRC"] = merged["_merge"].map(src_map)
    merged["IN_BASE1"] = merged["_merge"].isin(["left_only", "both"])
    merged["IN_BASE2"] = merged["_merge"].isin(["right_only", "both"])

    print(f"Fusion OUTER {year} (join_on_month={join_on_month}) → {len(merged):,} lignes, {merged.shape[1]} colonnes")
    print("\nBilan _merge :")
    print(merged["_merge"].value_counts(dropna=False))

    # 10) Exports non-appariés : XLSX si petit, sinon Parquet
    if export_unmatched_prefix:
        left_cols  = [c for c in ["YEAR","MONTH","MATRICULE","SOURCE_FICHIER"] if c in merged.columns]
        right_cols = [c for c in ["YEAR","MONTH","MATRICULE","DATE_COLLECTE"] if c in merged.columns]

        left_only_df  = merged.loc[merged["_merge"]=="left_only",  left_cols]
        right_only_df = merged.loc[merged["_merge"]=="right_only", right_cols]

        if len(left_only_df) > 0:
            if len(left_only_df) <= max_xlsx_rows:
                left_path = f"{export_unmatched_prefix}_{year}_left_only.xlsx"
                write_df_to_xlsx_chunked(left_only_df, left_path, sheet_base="left_only",
                                         chunk_rows=min(xlsx_chunk_rows, max(10_000, len(left_only_df))),
                                         engine=xlsx_engine)
                print("XLSX écrit :", left_path)
            else:
                left_path = f"{export_unmatched_prefix}_{year}_left_only.parquet"
                left_only_df.to_parquet(left_path, engine="pyarrow", index=False)
                print("Parquet écrit (plus léger) :", left_path)

        if len(right_only_df) > 0:
            if len(right_only_df) <= max_xlsx_rows:
                right_path = f"{export_unmatched_prefix}_{year}_right_only.xlsx"
                write_df_to_xlsx_chunked(right_only_df, right_path, sheet_base="right_only",
                                         chunk_rows=min(xlsx_chunk_rows, max(10_000, len(right_only_df))),
                                         engine=xlsx_engine)
                print("XLSX écrit :", right_path)
            else:
                right_path = f"{export_unmatched_prefix}_{year}_right_only.parquet"
                right_only_df.to_parquet(right_path, engine="pyarrow", index=False)
                print("Parquet écrit (plus léger) :", right_path)

    gc.collect()
    return merged


## 2.7 Faire une vérification pour voir si la sommme pour chaque ligne à problème donne le montant brute pour les colonnes répétés


In [8]:
def get_multi_filled_row_indices(
    df: pd.DataFrame,
    *,
    source_col: str = "SOURCE_FICHIER",
    treat_zero_as_filled: bool = True,
) -> pd.Index:
    if source_col not in df.columns:
        raise KeyError(f"Colonne {source_col!r} absente.")

    repeated = find_repeated_columns(df, exclude=(source_col,))
    if not repeated:
        return pd.Index([])

    years_series = extract_years_series(df[source_col])
    if years_series.isna().all():
        years_series = pd.Series(["ALL"] * len(df), index=df.index)

    bad_idx_sets: List[pd.Index] = []
    for base, cols in repeated.items():
        block = df[cols]
        filled = _filled_mask(block, treat_zero_as_filled)
        nb_filled = filled.sum(axis=1)

        for _, inds in years_series.groupby(years_series).groups.items():
            mask_year = df.index.isin(inds)
            viol_mask = (nb_filled > 1) & mask_year
            if viol_mask.any():
                bad_idx_sets.append(df.index[viol_mask])

    if not bad_idx_sets:
        return pd.Index([])

    bad_union = bad_idx_sets[0]
    for idx in bad_idx_sets[1:]:
        bad_union = bad_union.union(idx)
    return bad_union

def check_sum_all_base_vs_reference_on_index(
    df: pd.DataFrame,
    row_index_filter: pd.Index,
    *,
    reference_col: str = "MONTANT BRUT",
    abs_tol: float = 1e-6,
    rel_tol: float = 0.0,
    show_examples: int = 10,
) -> pd.DataFrame:
    if reference_col not in df.columns:
        raise KeyError(f"Colonne de référence {reference_col!r} absente.")

    if len(row_index_filter) == 0:
        print("✅ Aucune ligne à tester (aucune exclusivité violée).")
        return pd.DataFrame(columns=["sum_all", "reference", "abs_diff", "rel_diff"])

    sub = df.loc[row_index_filter]

    num_cols = sub.select_dtypes(include=["number"]).columns.tolist()
    ref_numeric = (pd.to_numeric(sub[reference_col], errors="coerce")
                   if reference_col not in num_cols else sub[reference_col])

    sum_all = sub[num_cols].sum(axis=1, min_count=1)
    abs_diff = (sum_all - ref_numeric).abs()
    rel_diff = abs_diff / ref_numeric.replace(0, pd.NA)
    bad_mask = (abs_diff > abs_tol) & (rel_diff.fillna(np.inf) > rel_tol)

    out = sub.copy()
    out["sum_all"] = sum_all
    out["reference"] = ref_numeric
    out["abs_diff"] = abs_diff
    out["rel_diff"] = rel_diff

    anomalies = out.loc[bad_mask].sort_values("abs_diff", ascending=False)
    print(f"🔎 Lignes testées: {len(sub):,} | Anomalies (somme ≠ {reference_col}): {len(anomalies):,}")
    if not anomalies.empty:
        print("\n📌 Exemples d'anomalies:")
        print(anomalies.head(show_examples).to_string(index=True))
    return anomalies


## 2.8 Enregistrement sur la plateforme 


In [9]:
def get_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=ENDPOINT_URL,
        aws_access_key_id=AWS_KEY,
        aws_secret_access_key=AWS_SECRET,
        verify=VERIFY_SSL,
    )

def save_parquet_to_s3(df, object_key: str, bucket: str = None, s3_client=None, engine: str = "pyarrow"):
    if bucket is None:
        bucket = globals().get("BUCKET")
        if not bucket:
            raise ValueError("BUCKET is not defined. Pass bucket=... or define a global BUCKET before calling.")
    if s3_client is None:
        s3_client = get_s3_client()
    buf = io.BytesIO()
    df.to_parquet(buf, engine=engine, index=False)
    buf.seek(0)
    s3_client.put_object(Bucket=bucket, Key=object_key, Body=buf)
    print(f"✅ Sauvegardé : s3://{bucket}/{object_key}")
    return f"s3://{bucket}/{object_key}"

def save_panel_solde_year_parquet(df, year: int, prefix: str = "Solde/Panel_solde_1_2_", bucket: str = None, s3_client=None):
    if bucket is None:
        bucket = globals().get("BUCKET")
        if not bucket:
            raise ValueError("BUCKET is not defined. Pass bucket=... or define a global BUCKET before calling.")
    key = f"{prefix}{year}.parquet"
    return save_parquet_to_s3(df, object_key=key, bucket=bucket, s3_client=s3_client)


## 2.9 Empilement des différentes bases enregistré sur la plateforme

In [10]:
def read_parquet_s3(object_key: str, bucket: str = None, s3_client=None) -> pd.DataFrame:
    if bucket is None:
        bucket = globals().get("BUCKET")
        if not bucket:
            raise ValueError("BUCKET n'est pas défini.")
    if s3_client is None:
        s3_client = get_s3_client()
    obj = s3_client.get_object(Bucket=bucket, Key=object_key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))

def stack_panel_solde_years(
    years: List[int],
    input_pattern: str = "Solde/Panel_solde_1_2_{year}.parquet",
    output_key: str | None = None,
    bucket: str = None,
    s3_client=None,
):
    if bucket is None:
        bucket = globals().get("BUCKET")
        if not bucket:
            raise ValueError("BUCKET n'est pas défini.")

    if s3_client is None:
        s3_client = get_s3_client()

    dfs = []
    for y in years:
        key = input_pattern.format(year=y)
        try:
            df = read_parquet_s3(key, bucket=bucket, s3_client=s3_client)
            df.columns = df.columns.str.strip().str.upper()
            dfs.append(df)
            print(f"✔️ Chargé : {key}  ({df.shape[0]} lignes, {df.shape[1]} colonnes)")
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            if code in ("NoSuchKey", "404", "NotFound"):
                print(f"⚠️ Manquant (ignoré) : {key}")
            else:
                print(f"⚠️ Erreur sur {key} (ignoré) : {e}")
        except Exception as e:
            print(f"⚠️ Erreur sur {key} (ignoré) : {e}")

    if not dfs:
        raise RuntimeError("Aucune table chargée. Vérifie les années/chemins.")

    df_final = pd.concat(dfs, axis=0, join="outer", ignore_index=True)
    print("Colonnes finales :", df_final.columns.tolist())
    print(f"Shape final : {df_final.shape[0]} lignes x {df_final.shape[1]} colonnes")

    if output_key is None:
        output_key = f"Solde/Panel_solde_1_2_{min(years)}_{max(years)}.parquet"

    save_parquet_to_s3(df_final, object_key=output_key, bucket=bucket, s3_client=s3_client)
    gc.collect()
    return df_final


# 3) DÉFINITION DES PARAMÈTRES

In [11]:
ENDPOINT_URL = "http://minio.mon-namespace.svc.cluster.local:80"
AWS_KEY      = "wer1Or8j7hXa3QGk2M3M"
AWS_SECRET   = "YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt"
VERIFY_SSL   = False

BUCKET       = "admindataanstat"
KEY_PANEL_2  = "Solde/panel_solde_df_2.parquet"          # panel (montants, primes…)
KEY_PANEL_1  = "Solde/panel_solde_df_1_code.parquet"     # infos admin / codes

# Pour la projection (réduire la RAM): adapte selon ton schéma
COLUMNS_PANEL_2_MIN = None  # None = tout ; ou liste p.ex. ["SOURCE_FICHIER","MATRICULE||CODE_ORGANISME","MONTANT BRUT", ...]
COLUMNS_PANEL_1_MIN = None  # idem

# 4) APPEL DES FONCTIONS / PIPELINE

## 4.1 Charger panel_solde_df (panel 2)

In [12]:
panel_solde_df = read_parquet_from_s3(
    BUCKET, KEY_PANEL_2,
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    verify=VERIFY_SSL,
    columns=COLUMNS_PANEL_2_MIN
)
print(f"panel_solde_df: {len(panel_solde_df):,} lignes, {panel_solde_df.shape[1]} colonnes")


panel_solde_df: 15,081,954 lignes, 190 colonnes


## 4.2 Afficher les bases répétées détectées

In [13]:
_ = report_repeated_columns(panel_solde_df)

🔎 Bases répétées détectées: 4
  • differentiel familial -> ['differentiel familial (code 221)', 'differentiel familial (code 422)']
  • indemnite de logement -> ['indemnite de logement (code 263)', 'indemnite de logement (code 271)', 'indemnite de logement (code 265)']
  • participation judicature -> ['participation judicature (code 440)', 'participation judicature (code 441)', 'participation judicature (code 442)', 'participation judicature (code 443)']
  • responsabilite tresor -> ['responsabilite tresor (code 487)', 'responsabilite tresor (code 493)']


## 4.3 Vérifier l'exclusivité (au plus une colonne renseignée par base & par année)

In [15]:
excl_summary = check_repeated_columns_exclusivity_by_year(
        panel_solde_df,
        source_col="SOURCE_FICHIER",
        treat_zero_as_filled=True,  # mets False si "0" doit compter comme vide
        show_examples=10            # nb de lignes fautives à afficher (échantillon)
    )
    
    # Voir le tableau récap si besoin :
    # excl_summary.head(20)


🚨 Exclusivité violée — base='indemnite de logement', année=2015 : 42 lignes en anomalie (sur 2278823).
MATRICULE||CODE_ORGANISME SOURCE_FICHIER indemnite de logement (code 263) indemnite de logement (code 271) indemnite de logement (code 265)
                350537E22    012015.xlsx                            40000                            40000                             None
                351478Q22    012015.xlsx                            40000                           280000                             None
                365120B42    012015.xlsx                            40000                            40000                             None
                382643D22    012015.xlsx                            40000                             5000                             None
                350537E22    022015.xlsx                            40000                            40000                             None
                351478Q22    022015.xlsx                

## 4.4 Faire une vérification pour voir si la sommme pour chaque ligne à problème donne le montant brute pour les colonnes répétés


In [14]:
bad_idx = get_multi_filled_row_indices(
    panel_solde_df,
    source_col="SOURCE_FICHIER",
    treat_zero_as_filled=True
)
print("Nombre de lignes multi-remplies (toutes bases/années confondues):", len(bad_idx))


Nombre de lignes multi-remplies (toutes bases/années confondues): 286


In [15]:

# 3) Ne sommer QUE sur ces lignes, et comparer à MONTANT BRUT
bilan_sum_all = check_sum_all_base_vs_reference_on_index(
    panel_solde_df,
    bad_idx,
    reference_col="MONTANT BRUT",
    abs_tol=1e-6,
    rel_tol=0.0,
    show_examples=10
)

🔎 Lignes testées: 286 | Anomalies (somme ≠ MONTANT BRUT): 0



## 4.5 Consolider les colonnes répétées (1ère non nulle) et supprimer les sources

Il faut sommer montant pour avoir une seule colonne si les colonnes répétés donnent le Montant brut 

In [14]:
panel_solde_df = consolidate_all_years_and_drop_sources(
    panel_solde_df,
    source_col="SOURCE_FICHIER",
    mode="first_non_null",        # "sum" possible mais plus gourmand
    sep=" | ",
    treat_zero_as_filled=True,
    new_name_fmt="{base}",
    sum_chunk=50                  # utilisé seulement si mode="sum"
)
print(f"Après consolidation: {len(panel_solde_df):,} lignes, {panel_solde_df.shape[1]} colonnes")

Après consolidation: 15,081,954 lignes, 183 colonnes


## 4.6 Contrôle rapide : plus de colonnes '(code ...)'

In [15]:
res_codes = [c for c in panel_solde_df.columns if "(code" in c.lower()]
print("Colonnes '(code ...)' restantes :", res_codes)   # doit être []

Colonnes '(code ...)' restantes : []


## 4.7 Charger panel_solde_df_1 (panel 1)

In [16]:
panel_solde_df_1 = read_parquet_from_s3(
    BUCKET, KEY_PANEL_1,
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    verify=VERIFY_SSL,
    columns=COLUMNS_PANEL_1_MIN
)
print(f"panel_solde_df_1: {len(panel_solde_df_1):,} lignes, {panel_solde_df_1.shape[1]} colonnes")


panel_solde_df_1: 15,627,963 lignes, 40 colonnes


**Remarque**

La base panel_solde_df_1 a  15,627,963 lignes alors  que la base panel_solde_df a 15,081,954 lignes

## 4.8 Fusion sur une année donnée Et Enregistrement des non matchs dans un fichier excel

### 4.8.1 Fusion sur l'année 2015

In [17]:
# ----------------------- 4.3 FUSION & SAUVEGARDE ----------------------

# === Modes d'exécution ===
PROCESS_MODE        = "single_year"      # "single_year" (début) ou "loop"
SAFE_ONLY_YEAR      = 2015               # année cible en mode "single_year"
YEARS_TO_PROCESS    = list(range(2015, 2016))  # utilisé si PROCESS_MODE="loop"

# === Exports (éviter l'XLSX géant) ===
DISABLE_XLSX    = True                   # True = pas de XLSX ; False = XLSX plafonné
MAX_XLSX_ROWS   = 200_000                # au-delà -> export en Parquet
XLSX_CHUNK_ROWS = 1_000_000
XLSX_ENGINE     = "xlsxwriter"

# === Colonnes à conserver ===
# Mettre KEEP_ALL_COLUMNS=True pour garder TOUTES les colonnes de gauche + droite (recommandé pour avoir ~223 colonnes)
# Sinon, mettre False et renseigner KEEP_LEFT_COLS / KEEP_RIGHT_COLS pour une projection sélective.
KEEP_ALL_COLUMNS = True

# Si KEEP_ALL_COLUMNS=False, renseigne tes listes (exemples ci-dessous)
KEEP_LEFT_COLS  = ["SOURCE_FICHIER", "MONTANT BRUT"]   # panel 2 (gauche) - ajoute ce dont tu as besoin
KEEP_RIGHT_COLS = ["DATE_COLLECTE", "ENTITE", "GRADE"] # panel 1 (droite) - ajoute ce dont tu as besoin

# (Option) Exclusions ciblées si certaines colonnes texte sont énormes (s'applique seulement si KEEP_ALL_COLUMNS=True)
EXCLUDE_LEFT_COLS  = []   # ex: ["COMMENTAIRE_LONG", "BLOB_DOC"]
EXCLUDE_RIGHT_COLS = []   # ex: ["JUSTIF_PDF_BASE64"]

# ---------------------------------------------------------------------

s3_client = get_s3_client()

def process_one_year(y: int):
    # Prépare les paramètres keep_* en fonction du mode choisi
    if KEEP_ALL_COLUMNS:
        # TOUT garder → keep_left/keep_right = None (aucune projection)
        keep_left  = None
        keep_right = None

        # Si tu veux exclure quelques colonnes lourdes malgré tout :
        # on transforme None → liste complète moins exclusions
        if EXCLUDE_LEFT_COLS or EXCLUDE_RIGHT_COLS:
            # Liste complète actuelle (dans l'état après consolidation)
            cols_left  = [c for c in panel_solde_df.columns if c not in EXCLUDE_LEFT_COLS]
            cols_right = [c for c in panel_solde_df_1.columns if c not in EXCLUDE_RIGHT_COLS]
            keep_left, keep_right = cols_left, cols_right
    else:
        # Projection sélective (listes définies plus haut)
        keep_left, keep_right = KEEP_LEFT_COLS, KEEP_RIGHT_COLS

    merged = merge_union_for_year_with_suffix(
        panel_solde_df, panel_solde_df_1,
        year=y,
        join_on_month=True,                  # clé = (YEAR, MONTH, MATRICULE)
        suffix_left="_b0",
        suffix_right="_b1",
        dedup_right=True,
        export_unmatched_prefix=None if DISABLE_XLSX else "bilan_merge",
        xlsx_chunk_rows=XLSX_CHUNK_ROWS,
        xlsx_engine=XLSX_ENGINE,
        keep_left=keep_left,                 # <-- défini ci-dessus
        keep_right=keep_right,               # <-- défini ci-dessus
        max_xlsx_rows=MAX_XLSX_ROWS
    )
    print("Colonnes fusionnées :", merged.shape[1])  # pour vérifier rapidement
    save_panel_solde_year_parquet(merged, year=y, s3_client=s3_client)
    del merged; gc.collect()

if PROCESS_MODE == "single_year":
    process_one_year(SAFE_ONLY_YEAR)

elif PROCESS_MODE == "loop":
    for y in YEARS_TO_PROCESS:
        print(f"\n===== TRAITEMENT ANNÉE {y} =====")
        process_one_year(y)


Fusion OUTER 2015 (join_on_month=True) → 2,305,311 lignes, 229 colonnes

Bilan _merge :
_merge
both          2278182
right_only      26488
left_only         641
Name: count, dtype: int64
Colonnes fusionnées : 229
✅ Sauvegardé : s3://admindataanstat/Solde/Panel_solde_1_2_2015.parquet


### 4.8.2 Fusion sur l'année 2016

In [ ]:
# ----------------------- 4.3 FUSION & SAUVEGARDE ----------------------

# === Modes d'exécution ===
PROCESS_MODE        = "single_year"      # "single_year" (début) ou "loop"
SAFE_ONLY_YEAR      = 2016              # année cible en mode "single_year"
YEARS_TO_PROCESS    = list(range(2016, 2017))  # utilisé si PROCESS_MODE="loop"

# === Exports (éviter l'XLSX géant) ===
DISABLE_XLSX    = True                   # True = pas de XLSX ; False = XLSX plafonné
MAX_XLSX_ROWS   = 200_000                # au-delà -> export en Parquet
XLSX_CHUNK_ROWS = 1_000_000
XLSX_ENGINE     = "xlsxwriter"

# === Colonnes à conserver ===
# Mettre KEEP_ALL_COLUMNS=True pour garder TOUTES les colonnes de gauche + droite (recommandé pour avoir ~223 colonnes)
# Sinon, mettre False et renseigner KEEP_LEFT_COLS / KEEP_RIGHT_COLS pour une projection sélective.
KEEP_ALL_COLUMNS = True

# Si KEEP_ALL_COLUMNS=False, renseigne tes listes (exemples ci-dessous)
KEEP_LEFT_COLS  = ["SOURCE_FICHIER", "MONTANT BRUT"]   # panel 2 (gauche) - ajoute ce dont tu as besoin
KEEP_RIGHT_COLS = ["DATE_COLLECTE", "ENTITE", "GRADE"] # panel 1 (droite) - ajoute ce dont tu as besoin

# (Option) Exclusions ciblées si certaines colonnes texte sont énormes (s'applique seulement si KEEP_ALL_COLUMNS=True)
EXCLUDE_LEFT_COLS  = []   # ex: ["COMMENTAIRE_LONG", "BLOB_DOC"]
EXCLUDE_RIGHT_COLS = []   # ex: ["JUSTIF_PDF_BASE64"]

# ---------------------------------------------------------------------

s3_client = get_s3_client()

def process_one_year(y: int):
    # Prépare les paramètres keep_* en fonction du mode choisi
    if KEEP_ALL_COLUMNS:
        # TOUT garder → keep_left/keep_right = None (aucune projection)
        keep_left  = None
        keep_right = None

        # Si tu veux exclure quelques colonnes lourdes malgré tout :
        # on transforme None → liste complète moins exclusions
        if EXCLUDE_LEFT_COLS or EXCLUDE_RIGHT_COLS:
            # Liste complète actuelle (dans l'état après consolidation)
            cols_left  = [c for c in panel_solde_df.columns if c not in EXCLUDE_LEFT_COLS]
            cols_right = [c for c in panel_solde_df_1.columns if c not in EXCLUDE_RIGHT_COLS]
            keep_left, keep_right = cols_left, cols_right
    else:
        # Projection sélective (listes définies plus haut)
        keep_left, keep_right = KEEP_LEFT_COLS, KEEP_RIGHT_COLS

    merged = merge_union_for_year_with_suffix(
        panel_solde_df, panel_solde_df_1,
        year=y,
        join_on_month=True,                  # clé = (YEAR, MONTH, MATRICULE)
        suffix_left="_b0",
        suffix_right="_b1",
        dedup_right=True,
        export_unmatched_prefix=None if DISABLE_XLSX else "bilan_merge",
        xlsx_chunk_rows=XLSX_CHUNK_ROWS,
        xlsx_engine=XLSX_ENGINE,
        keep_left=keep_left,                 # <-- défini ci-dessus
        keep_right=keep_right,               # <-- défini ci-dessus
        max_xlsx_rows=MAX_XLSX_ROWS
    )
    print("Colonnes fusionnées :", merged.shape[1])  # pour vérifier rapidement
    save_panel_solde_year_parquet(merged, year=y, s3_client=s3_client)
    del merged; gc.collect()

if PROCESS_MODE == "single_year":
    process_one_year(SAFE_ONLY_YEAR)

elif PROCESS_MODE == "loop":
    for y in YEARS_TO_PROCESS:
        print(f"\n===== TRAITEMENT ANNÉE {y} =====")
        process_one_year(y)


### 4.8.3 Fusion sur l'année 2017

In [17]:
# ----------------------- 4.3 FUSION & SAUVEGARDE ----------------------

# === Modes d'exécution ===
PROCESS_MODE        = "single_year"      # "single_year" (début) ou "loop"
SAFE_ONLY_YEAR      = 2017               # année cible en mode "single_year"
YEARS_TO_PROCESS    = list(range(2017, 2018))  # utilisé si PROCESS_MODE="loop"

# === Exports (éviter l'XLSX géant) ===
DISABLE_XLSX    = True                   # True = pas de XLSX ; False = XLSX plafonné
MAX_XLSX_ROWS   = 200_000                # au-delà -> export en Parquet
XLSX_CHUNK_ROWS = 1_000_000
XLSX_ENGINE     = "xlsxwriter"

# === Colonnes à conserver ===
# Mettre KEEP_ALL_COLUMNS=True pour garder TOUTES les colonnes de gauche + droite (recommandé pour avoir ~223 colonnes)
# Sinon, mettre False et renseigner KEEP_LEFT_COLS / KEEP_RIGHT_COLS pour une projection sélective.
KEEP_ALL_COLUMNS = True

# Si KEEP_ALL_COLUMNS=False, renseigne tes listes (exemples ci-dessous)
KEEP_LEFT_COLS  = ["SOURCE_FICHIER", "MONTANT BRUT"]   # panel 2 (gauche) - ajoute ce dont tu as besoin
KEEP_RIGHT_COLS = ["DATE_COLLECTE", "ENTITE", "GRADE"] # panel 1 (droite) - ajoute ce dont tu as besoin

# (Option) Exclusions ciblées si certaines colonnes texte sont énormes (s'applique seulement si KEEP_ALL_COLUMNS=True)
EXCLUDE_LEFT_COLS  = []   # ex: ["COMMENTAIRE_LONG", "BLOB_DOC"]
EXCLUDE_RIGHT_COLS = []   # ex: ["JUSTIF_PDF_BASE64"]

# ---------------------------------------------------------------------

s3_client = get_s3_client()

def process_one_year(y: int):
    # Prépare les paramètres keep_* en fonction du mode choisi
    if KEEP_ALL_COLUMNS:
        # TOUT garder → keep_left/keep_right = None (aucune projection)
        keep_left  = None
        keep_right = None

        # Si tu veux exclure quelques colonnes lourdes malgré tout :
        # on transforme None → liste complète moins exclusions
        if EXCLUDE_LEFT_COLS or EXCLUDE_RIGHT_COLS:
            # Liste complète actuelle (dans l'état après consolidation)
            cols_left  = [c for c in panel_solde_df.columns if c not in EXCLUDE_LEFT_COLS]
            cols_right = [c for c in panel_solde_df_1.columns if c not in EXCLUDE_RIGHT_COLS]
            keep_left, keep_right = cols_left, cols_right
    else:
        # Projection sélective (listes définies plus haut)
        keep_left, keep_right = KEEP_LEFT_COLS, KEEP_RIGHT_COLS

    merged = merge_union_for_year_with_suffix(
        panel_solde_df, panel_solde_df_1,
        year=y,
        join_on_month=True,                  # clé = (YEAR, MONTH, MATRICULE)
        suffix_left="_b0",
        suffix_right="_b1",
        dedup_right=True,
        export_unmatched_prefix=None if DISABLE_XLSX else "bilan_merge",
        xlsx_chunk_rows=XLSX_CHUNK_ROWS,
        xlsx_engine=XLSX_ENGINE,
        keep_left=keep_left,                 # <-- défini ci-dessus
        keep_right=keep_right,               # <-- défini ci-dessus
        max_xlsx_rows=MAX_XLSX_ROWS
    )
    print("Colonnes fusionnées :", merged.shape[1])  # pour vérifier rapidement
    save_panel_solde_year_parquet(merged, year=y, s3_client=s3_client)
    del merged; gc.collect()

if PROCESS_MODE == "single_year":
    process_one_year(SAFE_ONLY_YEAR)

elif PROCESS_MODE == "loop":
    for y in YEARS_TO_PROCESS:
        print(f"\n===== TRAITEMENT ANNÉE {y} =====")
        process_one_year(y)


Fusion OUTER 2017 (join_on_month=True) → 2,510,492 lignes, 229 colonnes

Bilan _merge :
_merge
both          2474710
right_only      34886
left_only         896
Name: count, dtype: int64
Colonnes fusionnées : 229
✅ Sauvegardé : s3://admindataanstat/Solde/Panel_solde_1_2_2017.parquet


### 4.8.4 Fusion sur l'année 2018

In [ ]:
# ----------------------- 4.3 FUSION & SAUVEGARDE ----------------------

# === Modes d'exécution ===
PROCESS_MODE        = "single_year"      # "single_year" (début) ou "loop"
SAFE_ONLY_YEAR      = 2018               # année cible en mode "single_year"
YEARS_TO_PROCESS    = list(range(2018, 2019))  # utilisé si PROCESS_MODE="loop"

# === Exports (éviter l'XLSX géant) ===
DISABLE_XLSX    = True                   # True = pas de XLSX ; False = XLSX plafonné
MAX_XLSX_ROWS   = 200_000                # au-delà -> export en Parquet
XLSX_CHUNK_ROWS = 1_000_000
XLSX_ENGINE     = "xlsxwriter"

# === Colonnes à conserver ===
# Mettre KEEP_ALL_COLUMNS=True pour garder TOUTES les colonnes de gauche + droite (recommandé pour avoir ~223 colonnes)
# Sinon, mettre False et renseigner KEEP_LEFT_COLS / KEEP_RIGHT_COLS pour une projection sélective.
KEEP_ALL_COLUMNS = True

# Si KEEP_ALL_COLUMNS=False, renseigne tes listes (exemples ci-dessous)
KEEP_LEFT_COLS  = ["SOURCE_FICHIER", "MONTANT BRUT"]   # panel 2 (gauche) - ajoute ce dont tu as besoin
KEEP_RIGHT_COLS = ["DATE_COLLECTE", "ENTITE", "GRADE"] # panel 1 (droite) - ajoute ce dont tu as besoin

# (Option) Exclusions ciblées si certaines colonnes texte sont énormes (s'applique seulement si KEEP_ALL_COLUMNS=True)
EXCLUDE_LEFT_COLS  = []   # ex: ["COMMENTAIRE_LONG", "BLOB_DOC"]
EXCLUDE_RIGHT_COLS = []   # ex: ["JUSTIF_PDF_BASE64"]

# ---------------------------------------------------------------------

s3_client = get_s3_client()

def process_one_year(y: int):
    # Prépare les paramètres keep_* en fonction du mode choisi
    if KEEP_ALL_COLUMNS:
        # TOUT garder → keep_left/keep_right = None (aucune projection)
        keep_left  = None
        keep_right = None

        # Si tu veux exclure quelques colonnes lourdes malgré tout :
        # on transforme None → liste complète moins exclusions
        if EXCLUDE_LEFT_COLS or EXCLUDE_RIGHT_COLS:
            # Liste complète actuelle (dans l'état après consolidation)
            cols_left  = [c for c in panel_solde_df.columns if c not in EXCLUDE_LEFT_COLS]
            cols_right = [c for c in panel_solde_df_1.columns if c not in EXCLUDE_RIGHT_COLS]
            keep_left, keep_right = cols_left, cols_right
    else:
        # Projection sélective (listes définies plus haut)
        keep_left, keep_right = KEEP_LEFT_COLS, KEEP_RIGHT_COLS

    merged = merge_union_for_year_with_suffix(
        panel_solde_df, panel_solde_df_1,
        year=y,
        join_on_month=True,                  # clé = (YEAR, MONTH, MATRICULE)
        suffix_left="_b0",
        suffix_right="_b1",
        dedup_right=True,
        export_unmatched_prefix=None if DISABLE_XLSX else "bilan_merge",
        xlsx_chunk_rows=XLSX_CHUNK_ROWS,
        xlsx_engine=XLSX_ENGINE,
        keep_left=keep_left,                 # <-- défini ci-dessus
        keep_right=keep_right,               # <-- défini ci-dessus
        max_xlsx_rows=MAX_XLSX_ROWS
    )
    print("Colonnes fusionnées :", merged.shape[1])  # pour vérifier rapidement
    save_panel_solde_year_parquet(merged, year=y, s3_client=s3_client)
    del merged; gc.collect()

if PROCESS_MODE == "single_year":
    process_one_year(SAFE_ONLY_YEAR)

elif PROCESS_MODE == "loop":
    for y in YEARS_TO_PROCESS:
        print(f"\n===== TRAITEMENT ANNÉE {y} =====")
        process_one_year(y)


### 4.8.5 Fusion sur l'année 2019

In [ ]:
# ----------------------- 4.3 FUSION & SAUVEGARDE ----------------------

# === Modes d'exécution ===
PROCESS_MODE        = "single_year"      # "single_year" (début) ou "loop"
SAFE_ONLY_YEAR      = 2019               # année cible en mode "single_year"
YEARS_TO_PROCESS    = list(range(2019, 2020))  # utilisé si PROCESS_MODE="loop"

# === Exports (éviter l'XLSX géant) ===
DISABLE_XLSX    = True                   # True = pas de XLSX ; False = XLSX plafonné
MAX_XLSX_ROWS   = 200_000                # au-delà -> export en Parquet
XLSX_CHUNK_ROWS = 1_000_000
XLSX_ENGINE     = "xlsxwriter"

# === Colonnes à conserver ===
# Mettre KEEP_ALL_COLUMNS=True pour garder TOUTES les colonnes de gauche + droite (recommandé pour avoir ~223 colonnes)
# Sinon, mettre False et renseigner KEEP_LEFT_COLS / KEEP_RIGHT_COLS pour une projection sélective.
KEEP_ALL_COLUMNS = True

# Si KEEP_ALL_COLUMNS=False, renseigne tes listes (exemples ci-dessous)
KEEP_LEFT_COLS  = ["SOURCE_FICHIER", "MONTANT BRUT"]   # panel 2 (gauche) - ajoute ce dont tu as besoin
KEEP_RIGHT_COLS = ["DATE_COLLECTE", "ENTITE", "GRADE"] # panel 1 (droite) - ajoute ce dont tu as besoin

# (Option) Exclusions ciblées si certaines colonnes texte sont énormes (s'applique seulement si KEEP_ALL_COLUMNS=True)
EXCLUDE_LEFT_COLS  = []   # ex: ["COMMENTAIRE_LONG", "BLOB_DOC"]
EXCLUDE_RIGHT_COLS = []   # ex: ["JUSTIF_PDF_BASE64"]

# ---------------------------------------------------------------------

s3_client = get_s3_client()

def process_one_year(y: int):
    # Prépare les paramètres keep_* en fonction du mode choisi
    if KEEP_ALL_COLUMNS:
        # TOUT garder → keep_left/keep_right = None (aucune projection)
        keep_left  = None
        keep_right = None

        # Si tu veux exclure quelques colonnes lourdes malgré tout :
        # on transforme None → liste complète moins exclusions
        if EXCLUDE_LEFT_COLS or EXCLUDE_RIGHT_COLS:
            # Liste complète actuelle (dans l'état après consolidation)
            cols_left  = [c for c in panel_solde_df.columns if c not in EXCLUDE_LEFT_COLS]
            cols_right = [c for c in panel_solde_df_1.columns if c not in EXCLUDE_RIGHT_COLS]
            keep_left, keep_right = cols_left, cols_right
    else:
        # Projection sélective (listes définies plus haut)
        keep_left, keep_right = KEEP_LEFT_COLS, KEEP_RIGHT_COLS

    merged = merge_union_for_year_with_suffix(
        panel_solde_df, panel_solde_df_1,
        year=y,
        join_on_month=True,                  # clé = (YEAR, MONTH, MATRICULE)
        suffix_left="_b0",
        suffix_right="_b1",
        dedup_right=True,
        export_unmatched_prefix=None if DISABLE_XLSX else "bilan_merge",
        xlsx_chunk_rows=XLSX_CHUNK_ROWS,
        xlsx_engine=XLSX_ENGINE,
        keep_left=keep_left,                 # <-- défini ci-dessus
        keep_right=keep_right,               # <-- défini ci-dessus
        max_xlsx_rows=MAX_XLSX_ROWS
    )
    print("Colonnes fusionnées :", merged.shape[1])  # pour vérifier rapidement
    save_panel_solde_year_parquet(merged, year=y, s3_client=s3_client)
    del merged; gc.collect()

if PROCESS_MODE == "single_year":
    process_one_year(SAFE_ONLY_YEAR)

elif PROCESS_MODE == "loop":
    for y in YEARS_TO_PROCESS:
        print(f"\n===== TRAITEMENT ANNÉE {y} =====")
        process_one_year(y)


### 4.8.6 Fusion sur l'année 2020

In [17]:
# ----------------------- 4.3 FUSION & SAUVEGARDE ----------------------

# === Modes d'exécution ===
PROCESS_MODE        = "single_year"      # "single_year" (début) ou "loop"
SAFE_ONLY_YEAR      = 2020             # année cible en mode "single_year"
YEARS_TO_PROCESS    = list(range(2020, 2021))  # utilisé si PROCESS_MODE="loop"

# === Exports (éviter l'XLSX géant) ===
DISABLE_XLSX    = True                   # True = pas de XLSX ; False = XLSX plafonné
MAX_XLSX_ROWS   = 200_000                # au-delà -> export en Parquet
XLSX_CHUNK_ROWS = 1_000_000
XLSX_ENGINE     = "xlsxwriter"

# === Colonnes à conserver ===
# Mettre KEEP_ALL_COLUMNS=True pour garder TOUTES les colonnes de gauche + droite (recommandé pour avoir ~223 colonnes)
# Sinon, mettre False et renseigner KEEP_LEFT_COLS / KEEP_RIGHT_COLS pour une projection sélective.
KEEP_ALL_COLUMNS = True

# Si KEEP_ALL_COLUMNS=False, renseigne tes listes (exemples ci-dessous)
KEEP_LEFT_COLS  = ["SOURCE_FICHIER", "MONTANT BRUT"]   # panel 2 (gauche) - ajoute ce dont tu as besoin
KEEP_RIGHT_COLS = ["DATE_COLLECTE", "ENTITE", "GRADE"] # panel 1 (droite) - ajoute ce dont tu as besoin

# (Option) Exclusions ciblées si certaines colonnes texte sont énormes (s'applique seulement si KEEP_ALL_COLUMNS=True)
EXCLUDE_LEFT_COLS  = []   # ex: ["COMMENTAIRE_LONG", "BLOB_DOC"]
EXCLUDE_RIGHT_COLS = []   # ex: ["JUSTIF_PDF_BASE64"]

# ---------------------------------------------------------------------

s3_client = get_s3_client()

def process_one_year(y: int):
    # Prépare les paramètres keep_* en fonction du mode choisi
    if KEEP_ALL_COLUMNS:
        # TOUT garder → keep_left/keep_right = None (aucune projection)
        keep_left  = None
        keep_right = None

        # Si tu veux exclure quelques colonnes lourdes malgré tout :
        # on transforme None → liste complète moins exclusions
        if EXCLUDE_LEFT_COLS or EXCLUDE_RIGHT_COLS:
            # Liste complète actuelle (dans l'état après consolidation)
            cols_left  = [c for c in panel_solde_df.columns if c not in EXCLUDE_LEFT_COLS]
            cols_right = [c for c in panel_solde_df_1.columns if c not in EXCLUDE_RIGHT_COLS]
            keep_left, keep_right = cols_left, cols_right
    else:
        # Projection sélective (listes définies plus haut)
        keep_left, keep_right = KEEP_LEFT_COLS, KEEP_RIGHT_COLS

    merged = merge_union_for_year_with_suffix(
        panel_solde_df, panel_solde_df_1,
        year=y,
        join_on_month=True,                  # clé = (YEAR, MONTH, MATRICULE)
        suffix_left="_b0",
        suffix_right="_b1",
        dedup_right=True,
        export_unmatched_prefix=None if DISABLE_XLSX else "bilan_merge",
        xlsx_chunk_rows=XLSX_CHUNK_ROWS,
        xlsx_engine=XLSX_ENGINE,
        keep_left=keep_left,                 # <-- défini ci-dessus
        keep_right=keep_right,               # <-- défini ci-dessus
        max_xlsx_rows=MAX_XLSX_ROWS
    )
    print("Colonnes fusionnées :", merged.shape[1])  # pour vérifier rapidement
    save_panel_solde_year_parquet(merged, year=y, s3_client=s3_client)
    del merged; gc.collect()

if PROCESS_MODE == "single_year":
    process_one_year(SAFE_ONLY_YEAR)

elif PROCESS_MODE == "loop":
    for y in YEARS_TO_PROCESS:
        print(f"\n===== TRAITEMENT ANNÉE {y} =====")
        process_one_year(y)


Fusion OUTER 2020 (join_on_month=True) → 2,922,599 lignes, 229 colonnes

Bilan _merge :
_merge
both          2841349
right_only      80124
left_only        1126
Name: count, dtype: int64
Colonnes fusionnées : 229
✅ Sauvegardé : s3://admindataanstat/Solde/Panel_solde_1_2_2020.parquet


## 4.9 Empilement 

In [18]:
# ===== 4.9 Empilement multi-années (simple) =====
s3_client = get_s3_client()
df_empile = stack_panel_solde_years(
    years=[2015, 2016, 2017, 2018, 2019, 2020],
    # output_key=None -> génère automatiquement Solde/Panel_solde_1_2_2015_2020.parquet
    s3_client=s3_client,
)
print("Empilement terminé :", df_empile.shape)


✔️ Chargé : Solde/Panel_solde_1_2_2015.parquet  (2305311 lignes, 229 colonnes)
✔️ Chargé : Solde/Panel_solde_1_2_2016.parquet  (2428508 lignes, 229 colonnes)
✔️ Chargé : Solde/Panel_solde_1_2_2017.parquet  (2510492 lignes, 229 colonnes)
✔️ Chargé : Solde/Panel_solde_1_2_2018.parquet  (2669894 lignes, 229 colonnes)
✔️ Chargé : Solde/Panel_solde_1_2_2019.parquet  (2795680 lignes, 229 colonnes)
✔️ Chargé : Solde/Panel_solde_1_2_2020.parquet  (2922599 lignes, 229 colonnes)


/tmp/ipykernel_129/390716893.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(dfs, axis=0, join="outer", ignore_index=True)
/tmp/ipykernel_129/390716893.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(dfs, axis=0, join="outer", ignore_index=True)
/tmp/ipykernel_129/390716893.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA co

Colonnes finales : ['SOLDE DE BASE', 'SALAIRE D AGENT TEMPORAIRE', 'SALAIRE DE GENS DE MAISON', 'FORFAIT INDICIAIRE AMBASSADEUR', 'COMPENSATRICE SMIG', 'REVALORISATION 2014', 'COMPENSATRICE NON IMP AMBASSADES', 'RISQUE SANITAIRE', 'COMPENSATRICE NON IMPOSABLE', 'COMPENSATRICE IMPOSABLE', 'COMPLEMENT INP HB', 'SUJETION GREFFIER EN CHEF', 'PRIME DE RECHERCHE', 'COMPENSATRICE AMBASSADEUR', 'CONTRAINTE', 'HEURES SUPPL , PERSONNEL ENSEIGNANT', 'HEURES DE COURS', 'PATROUILLE', 'TRAITEMENT PERSONNEL NON CLASSE', 'HEURES SUPPLEMENTAIRES ADMINISTRATION', 'FORFAIT CAP', 'TRAITEMENT CHEF DE CANTON', 'UTILISATION VEHICULE P/PPNAT', 'RISQUE', 'VERIFICATION CHAMBRE CPTES/POINCON OR', 'TRANSPORT', 'PRIME DE CAISSE', 'ENTRETIEN UNIFORME', 'ACQUISITION UNIFORME', 'DEPLACEMENT TEMPORAIRE', 'ALLOCATION ELEVE', 'TRAITEMENT AMBASSADEUR', 'INDEMNITE INGENIEUR SYSTEME', 'REGULARISATION CN EXERCICE ANTERIEUR', 'REGULARISATION IGR EXERCICE ANTERIEUR', 'REGULARISATION IGR', 'REGULARISATION CN', 'REGULARISATION 